# v0.14.5 — Connections and event loops

A SurrealDB WebSocket client is bound to the **event loop it connected on**. Until v0.14.5 the
connection manager cached a single client in a class attribute with no notion of that loop, so
handing it to a second loop failed with `got Future attached to a different loop` — or hung
outright ([issue #163](https://github.com/EulogySnowfall/SurrealDB-ORM-lite/issues/163)).

Two successive `asyncio.run(...)` calls in the same process were enough, because each one
creates and closes its own loop.

The cache is now keyed by the running loop: **one client per loop**.

Loop binding is an asyncio/SDK property, not a server one, so everything here behaves
identically on SurrealDB 2.6.x and 3.x.

In [1]:
import asyncio
import os
import threading

from surreal_orm_lite import BaseSurrealModel, SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")

SurrealDBConnectionManager.set_connection(
    url=f"ws://{HOST}:{PORT}/rpc",
    user="root",
    password="root",
    namespace="examples",
    database="examples",
)
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())

Connection configured: True


## 1. A fresh loop per call

Jupyter already owns a running event loop, so `asyncio.run()` cannot be called from a cell.
The helper below runs each coroutine in a worker thread instead — which is exactly what
`asyncio.run()` does anyway: build a loop, run, close it. It is also a real pattern in its own
right (a sync wrapper around async code, called from a thread pool).

In [2]:
def run_fresh_loop(coro):
    """Run `coro` on a brand-new event loop, in a worker thread."""
    box = {}

    def target():
        try:
            box["value"] = asyncio.run(coro)
        except BaseException as exc:  # noqa: BLE001 - surfaced below
            box["error"] = exc

    thread = threading.Thread(target=target)
    thread.start()
    thread.join(timeout=30)
    if "error" in box:
        raise box["error"]
    return box["value"]


async def ping() -> int:
    client = await SurrealDBConnectionManager.get_client()
    return await client.query("RETURN 1;")


print("loop 1:", run_fresh_loop(ping()))
print("loop 2:", run_fresh_loop(ping()))   # used to raise, or hang
print("loop 3:", run_fresh_loop(ping()))

loop 1: 1
loop 2: 1
loop 3: 1


Nothing above calls `close_connection()`. Requiring it before leaving a loop would only
move the trap somewhere less visible — a closed loop's entry is pruned automatically the next
time a client is requested.

## 2. The same thing through the ORM

In [3]:
class Note(BaseSurrealModel):
    id: str | None = None
    text: str


async def write_and_read(text: str) -> str:
    await Note.objects().delete_table()
    await Note(id="n1", text=text).save()
    row = await Note.objects().get("n1")
    return row.text


print("loop 1:", run_fresh_loop(write_and_read("hello")))
print("loop 2:", run_fresh_loop(write_and_read("again")))

loop 1: hello


loop 2: again


## 3. Two live loops keep their own client

A second loop must not evict the first one's connection while it is still in use. Below, a
worker thread takes a client and waits; this notebook's own loop then takes one; the worker
carries on afterwards. Two loops, two clients, neither disturbing the other.

In [4]:
results = {}
started, release = threading.Event(), threading.Event()


async def background() -> None:
    client = await SurrealDBConnectionManager.get_client()
    results["thread_client"] = id(client)
    started.set()
    # Wait until the notebook's loop has taken a client of its own.
    await asyncio.get_running_loop().run_in_executor(None, release.wait)
    results["thread_query"] = await client.query("RETURN 1;")
    await SurrealDBConnectionManager.close_connection()


thread = threading.Thread(target=lambda: asyncio.run(background()))
thread.start()
started.wait(timeout=10)

# This cell runs on the notebook's own loop — a second, live loop.
notebook_client = await SurrealDBConnectionManager.get_client()
results["notebook_client"] = id(notebook_client)
results["notebook_query"] = await notebook_client.query("RETURN 2;")

release.set()
thread.join(timeout=10)

print("distinct clients:", results["thread_client"] != results["notebook_client"])
print("notebook loop query:", results["notebook_query"])
print("thread query, after the notebook took its own client:", results["thread_query"])

distinct clients: True
notebook loop query: 2
thread query, after the notebook took its own client: 1


## 4. Closing is per loop

- `close_connection()` closes **the running loop's** client only — another live loop may still
  be using its own.
- `close_all_connections()` tears everything down.
- `is_connected()` answers for the loop asking; outside a loop it reports whether any loop
  still holds a client.

In [5]:
async def show() -> None:
    print("  before get_client:", SurrealDBConnectionManager.is_connected())
    await SurrealDBConnectionManager.get_client()
    print("  after  get_client:", SurrealDBConnectionManager.is_connected())
    await SurrealDBConnectionManager.close_connection()
    print("  after  close     :", SurrealDBConnectionManager.is_connected())


run_fresh_loop(show())
print("this notebook's loop still holds one:", SurrealDBConnectionManager.is_connected())

  before get_client: False
  after  get_client: True
  after  close     : False
this notebook's loop still holds one: True


## 5. Cleanup

In [6]:
await Note.objects().delete_table()
await SurrealDBConnectionManager.close_all_connections()
print("Cleaned up. Any client still cached:", SurrealDBConnectionManager.is_connected())

Cleaned up. Any client still cached: False
